In [3]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [4]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse


In [5]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/'
# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Mouse_chemical_gene
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_chemical_gene/Mouse_chemical_gene.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

In [24]:
# PubChem compound table: CID → IUPAC name and SMILES
Pubchem = pd.read_pickle(DB_DIR + 'databases_for_mapping/pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


In [7]:
Mouse = pd.read_csv(
    f'{BASE_DIR}data_collection/databases_for_mapping/ncbi/Mus_musculus.gene_info',
    sep='\t',
    
)
# Mouse["LocusTag"] = Mouse["LocusTag"].str.replace("Dmel_", "", regex=False)
Mouse
# Mouse_symbol2Locus_dict = dict(zip(Mouse['Symbol'], Mouse['LocusTag']))
# Mouse
# Mouse_symbol2Locus_dict

,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
0,10090,11287,Pzp,-,A1m|A2m|MAM,MGI:MGI:87854|Ensembl:ENSMUSG00000030359|Allia...,6,6 F3|6 63.02 cM,"PZP, alpha-2-macroglobulin like",protein-coding,Pzp,"PZP, alpha-2-macroglobulin like",O,pregnancy zone protein|alpha 1 macroglobulin|a...,20250308,-
1,10090,11298,Aanat,-,AA-NAT|Nat-2|Nat4|Snat,MGI:MGI:1328365|Ensembl:ENSMUSG00000020804|All...,11,11 81.43 cM|11 E2,arylalkylamine N-acetyltransferase,protein-coding,Aanat,arylalkylamine N-acetyltransferase,O,serotonin N-acetyltransferase|aralkylamine N-a...,20250308,-
2,10090,11302,Aatk,-,AATYK|aatyk1|mKIAA0641,MGI:MGI:1197518|Ensembl:ENSMUSG00000025375|All...,11,11 E2|11 83.96 cM,apoptosis-associated tyrosine kinase,protein-coding,Aatk,apoptosis-associated tyrosine kinase,O,serine/threonine-protein kinase LMTK1|apoptosi...,20250308,-
3,10090,11303,Abca1,-,ABC-1|Abc1,MGI:MGI:99607|Ensembl:ENSMUSG00000015243|Allia...,4,4 B2|4 28.57 cM,"ATP-binding cassette, sub-family A member 1",protein-coding,Abca1,"ATP-binding cassette, sub-family A member 1",O,phospholipid-transporting ATPase ABCA1|ATP-bin...,20250308,-
4,10090,11304,Abca4,-,Abc10|Abcr|D430003I15Rik|RmP,MGI:MGI:109424|Ensembl:ENSMUSG00000028125|Alli...,3,3 G1|3 52.94 cM,"ATP-binding cassette, sub-family A member 4",protein-coding,Abca4,"ATP-binding cassette, sub-family A member 4",O,retinal-specific phospholipid-transporting ATP...,20250308,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112364,57486,3337213,ND3,-,-,-,MT,-,NADH dehydrogenase subunit 3,protein-coding,-,-,-,-,20230818,-
112365,57486,3337214,ND1,-,-,-,MT,-,NADH dehydrogenase subunit 1,protein-coding,-,-,-,-,20230818,-
112366,57486,3337215,ND6,-,-,-,MT,-,NADH dehydrogenase subunit 6,protein-coding,-,-,-,-,20230818,-
112367,57486,3337216,ATP8,-,-,-,MT,-,ATP synthase F0 subunit 8,protein-coding,-,-,-,-,20230818,-


# stitch

In [13]:

stitch = pd.read_csv(PROC_DIR + 'stitch/stitch_MOUSE_CHEMICALENTITY_GENE.csv')

stitch.columns = stitch.columns.str.lower()
stitch['kg_type'] = 'Generalised'
stitch['kg_source'] = 'stitch'
stitch['species'] = 'M.musculus'
stitch['tail_id_is'] = 'NCBI_ID'

print(f"stitch: {len(stitch):,} rows")
stitch

stitch: 21,357,298 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species,kg_type
0,1,ChemicalEntity_Gene,Aanat,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,arylalkylamine N-acetyltransferase [Source:MGI...,M.musculus,Generalised
1,1,ChemicalEntity_Gene,Acaa1a,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-Coenzyme A acyltransferase 1A [Source:M...,M.musculus,Generalised
2,1,ChemicalEntity_Gene,Acaa1b,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-Coenzyme A acyltransferase 1B [Source:M...,M.musculus,Generalised
3,1,ChemicalEntity_Gene,Acaa2,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-CoA acyltransferase 2 [Source:MGI Symbo...,M.musculus,Generalised
4,1,ChemicalEntity_Gene,Acadm,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,"acyl-Coenzyme A dehydrogenase, medium chain [S...",M.musculus,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21357293,9999994,ChemicalEntity_Gene,Ces4a,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,carboxylesterase 4A [Source:MGI Symbol;Acc:MGI...,M.musculus,Generalised
21357294,9999994,ChemicalEntity_Gene,Nlgn1,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,neuroligin 1 [Source:MGI Symbol;Acc:MGI:2179435],M.musculus,Generalised
21357295,9999994,ChemicalEntity_Gene,Nlgn2,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,neuroligin 2 [Source:MGI Symbol;Acc:MGI:2681835],M.musculus,Generalised
21357296,9999994,ChemicalEntity_Gene,Nlgn3,ChemicalEntity,NaN,Gene,stitch,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,neuroligin 3 [Source:MGI Symbol;Acc:MGI:2444609],M.musculus,Generalised


# Consolidate into Unified Schem

In [14]:
# List all source DataFrames to include
source_dfs = [
    stitch 
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 21,357,298


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1,ChemicalEntity_Gene,Aanat,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,arylalkylamine N-acetyltransferase [Source:MGI...,M.musculus
1,1,ChemicalEntity_Gene,Acaa1a,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-Coenzyme A acyltransferase 1A [Source:M...,M.musculus
2,1,ChemicalEntity_Gene,Acaa1b,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-Coenzyme A acyltransferase 1B [Source:M...,M.musculus
3,1,ChemicalEntity_Gene,Acaa2,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-CoA acyltransferase 2 [Source:MGI Symbo...,M.musculus
4,1,ChemicalEntity_Gene,Acadm,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,"acyl-Coenzyme A dehydrogenase, medium chain [S...",M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21357293,9999994,ChemicalEntity_Gene,Ces4a,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,carboxylesterase 4A [Source:MGI Symbol;Acc:MGI...,M.musculus
21357294,9999994,ChemicalEntity_Gene,Nlgn1,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,neuroligin 1 [Source:MGI Symbol;Acc:MGI:2179435],M.musculus
21357295,9999994,ChemicalEntity_Gene,Nlgn2,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,neuroligin 2 [Source:MGI Symbol;Acc:MGI:2681835],M.musculus
21357296,9999994,ChemicalEntity_Gene,Nlgn3,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-[1-[(3-chlorophenyl)methyl]piperidin-4-yl]-1...,neuroligin 3 [Source:MGI Symbol;Acc:MGI:2444609],M.musculus


# Sanity Check — Distinct Values

In [15]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



kg_source           : {'stitch'}
head_id_is          : {'Pubchem'}
tail_id_is          : {'NCBI_ID'}


In [16]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 21,357,298 remaining


# NaN Audit (pre-dedup)

In [17]:
true_nan   = final_df.isna().sum()
genDR_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_genDR_count": genDR_nan,
    'Total_NaN_like':     true_nan + genDR_nan
})

,NaN_count,'NAN'_genDR_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,21357298,21357298,42714596
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [18]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 21,357,298  |  After dedup: 21,357,298


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,1,ChemicalEntity_Gene,Aanat,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,arylalkylamine N-acetyltransferase [Source:MGI...
1,1,ChemicalEntity_Gene,Acaa1a,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-Coenzyme A acyltransferase 1A [Source:M...
2,1,ChemicalEntity_Gene,Acaa1b,ChemicalEntity,NaN,Gene,stitch,Generalised,Pubchem,NCBI_ID,3-acetyloxy-4-(trimethylazaniumyl)butanoate,acetyl-Coenzyme A acyltransferase 1B [Source:M...


In [21]:
# final_df_dedup[final_df_dedup['head_detail_name'].isna()]

In [24]:
# # 1. Use .loc to target rows where 'head_detail_name' is NaN
# # 2. Map the 'CID' (or relevant key column) against your dictionary
# # 3. Assign it back to the 'head_detail_name' column

# mask = final_df_dedup['head_detail_name'].isna()

# final_df_dedup.loc[mask, 'head_detail_name'] = final_df_dedup.loc[mask, 'head_detail_name'].map(Pubchem_IUPAC_CID_Dict)
# final_df_dedup

In [25]:
true_nan   = final_df_dedup.isna().sum()
genDR_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_genDR_count": genDR_nan,
    'Total_NaN_like':     true_nan + genDR_nan
})

,NaN_count,'NAN'_genDR_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,21357298,21357298,42714596
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [26]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'stitch'} kg_source
stitch    21357298
Name: count, dtype: int64


In [27]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    21357298
Name: count, dtype: int64


In [28]:
# final_df_dedup[final_df_dedup['head_detail_name'].isna()]

In [29]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 21,357,298 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_chemical_gene/Mouse_chemical_gene.csv


In [20]:
#